# 2.1 Operations with Matrices

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 2:** Matrices

---

### What you will learn

- How to **add** matrices and **multiply** a matrix by a scalar.
- How to **multiply** two matrices using row-column dot products.
- Why matrix multiplication is **not commutative** in general.
- What the **identity matrix** is and why it acts as a multiplicative identity.
- How the **transpose** operation works and what properties it satisfies.

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.ops import add, scalar_mul, multiply, transpose
from linalg_core import EPSILON

import numpy as np
import matplotlib.pyplot as plt
import math

---

## 2. Motivation --- Factory Revenue

A company operates **two factories**, each producing three products.
Monthly output data is stored in matrices where rows represent factories
and columns represent products.

| | Product 1 | Product 2 | Product 3 |
|---|---|---|---|
| **Factory 1** | 120 | 80 | 40 |
| **Factory 2** | 100 | 60 | 50 |

If the selling prices are \$25, \$30, and \$15 per unit respectively,
what is the **total revenue** for each factory?

This is a matrix multiplication problem: if $A$ is the $2 \times 3$ output matrix
and $\mathbf{c}$ is the $3 \times 1$ cost vector, then the revenue vector is $A\mathbf{c}$.

In [ ]:
A_jan = Matrix.from_lists([
    [120, 80, 40],
    [100, 60, 50],
])

B_feb = Matrix.from_lists([
    [130, 70, 55],
    [110, 65, 45],
])

cost = Matrix.from_vector([25, 30, 15])

print("January production (A):")
print(A_jan)
print("\nFebruary production (B):")
print(B_feb)
print("\nCost vector (c):")
print(cost)

In [ ]:
revenue_jan = multiply(A_jan, cost)
print("January revenue by factory (A * c):")
print(revenue_jan)
print(f"\nFactory 1: ${revenue_jan[0,0]:,.0f}")
print(f"Factory 2: ${revenue_jan[1,0]:,.0f}")

We will revisit this factory example throughout the notebook as we build up
each matrix operation.

---

## 3. Build --- Core Concepts

We build up the fundamental matrix operations one at a time,
connecting each to the factory example where possible.

### 3.1 Matrix Addition

> **Definition (Larson, Sec. 2.1).** If $A$ and $B$ are $m \times n$ matrices, their
> **sum** $A + B$ is the $m \times n$ matrix whose $(i, j)$-entry is
>
> $$(A + B)_{ij} = a_{ij} + b_{ij}$$

Addition is defined **only** when both matrices have the same dimensions.
It is performed **entry by entry**.

**Factory interpretation:** Adding January and February production gives
total two-month output per factory per product.

In [ ]:
total_production = add(A_jan, B_feb)

print("January production (A):")
print(A_jan)
print("\nFebruary production (B):")
print(B_feb)
print("\nTotal two-month production (A + B):")
print(total_production)

### 3.2 Scalar Multiplication

> **Definition (Larson, Sec. 2.1).** If $A$ is an $m \times n$ matrix and $c$ is a scalar,
> the **scalar multiple** $cA$ is the $m \times n$ matrix whose $(i, j)$-entry is
>
> $$(cA)_{ij} = c \cdot a_{ij}$$

Every entry is multiplied by the same scalar.

**Factory interpretation:** If production is projected to **double** next quarter,
the projected output matrix is $2A$.

In [ ]:
doubled = scalar_mul(A_jan, 2)

print("January production (A):")
print(A_jan)
print("\nDoubled production (2A):")
print(doubled)

### 3.3 Definition: Matrix Multiplication

> **Definition (Larson, Sec. 2.1).** If $A$ is an $m \times n$ matrix and $B$ is an
> $n \times p$ matrix, their **product** $AB$ is the $m \times p$ matrix whose
> $(i, j)$-entry is the dot product of row $i$ of $A$ with column $j$ of $B$:
>
> $$(AB)_{ij} = \sum_{k=1}^{n} a_{ik} b_{kj}$$

**Key requirement:** The number of columns of $A$ must equal the number of rows of $B$.

$$\underbrace{A}_{m \times n} \cdot \underbrace{B}_{n \times p} = \underbrace{AB}_{m \times p}$$

The "inner" dimensions must match ($n = n$), and the result has the "outer" dimensions ($m \times p$).

### 3.4 Matrix Multiplication

Let us compute the January revenue $A \cdot \mathbf{c}$ step by step for the
first entry to see the dot product in action.

For Factory 1 revenue (row 0 of $A$ dotted with column 0 of $\mathbf{c}$):

$$(A\mathbf{c})_{0,0} = 120 \cdot 25 + 80 \cdot 30 + 40 \cdot 15 = 3000 + 2400 + 600 = 6000$$

In [ ]:
row_0 = A_jan.get_row(0)
col_0 = cost.get_col(0)

print("Row 0 of A:", row_0)
print("Col 0 of c:", col_0)
print()

dot = sum(a * b for a, b in zip(row_0, col_0))
print("Step-by-step dot product:")
terms = [f"{a:.0f} * {b:.0f}" for a, b in zip(row_0, col_0)]
print(" + ".join(terms), f"= {dot:.0f}")
print()

revenue = multiply(A_jan, cost)
print("Full result A * c:")
print(revenue)
print(f"\nVerification: entry (0,0) = {revenue[0,0]:.0f} matches dot product = {dot:.0f}")

### 3.5 Non-Commutativity of Matrix Multiplication

One of the most important differences between matrix multiplication and
scalar multiplication is that **order matters**. In general,

$$AB \neq BA$$

This is easy to demonstrate with two $2 \times 2$ matrices.

In [ ]:
P = Matrix.from_lists([[1, 2], [3, 4]])
Q = Matrix.from_lists([[5, 6], [7, 8]])

PQ = multiply(P, Q)
QP = multiply(Q, P)

print("P =")
print(P)
print("\nQ =")
print(Q)
print("\nPQ =")
print(PQ)
print("\nQP =")
print(QP)
print(f"\nPQ == QP? {PQ == QP}")

Even when both products are defined (e.g., both matrices are square),
the results are generally different. This is a fundamental property
that distinguishes matrix algebra from ordinary arithmetic.

### 3.6 The Identity Matrix

> **Definition (Larson, Sec. 2.1).** The $n \times n$ **identity matrix** $I_n$
> has 1s on the main diagonal and 0s elsewhere:
>
> $$(I_n)_{ij} = \begin{cases} 1 & \text{if } i = j \\ 0 & \text{if } i \neq j \end{cases}$$

The identity matrix is the **multiplicative identity** for matrix multiplication:

$$AI = IA = A$$

for any appropriately sized matrix $A$.

In [ ]:
I3 = Matrix.identity(3)
print("I_3 =")
print(I3)

M = Matrix.from_lists([
    [2, 5, 1],
    [7, 3, 9],
    [4, 6, 8],
])

MI = multiply(M, I3)
IM = multiply(I3, M)

print("\nM =")
print(M)
print("\nM * I =")
print(MI)
print("\nI * M =")
print(IM)
print(f"\nM * I == M? {MI == M}")
print(f"I * M == M? {IM == M}")

### 3.7 The Transpose

> **Definition (Larson, Sec. 2.1).** The **transpose** of an $m \times n$ matrix $A$,
> denoted $A^T$, is the $n \times m$ matrix whose $(i, j)$-entry is
>
> $$(A^T)_{ij} = a_{ji}$$

In other words, **row $i$ of $A$ becomes column $i$ of $A^T$**.
The transpose "flips" the matrix across its main diagonal.

In [ ]:
print("A (2x3):")
print(A_jan)

AT = transpose(A_jan)
print(f"\nA^T ({AT.rows}x{AT.cols}):")
print(AT)

print("\nRow 0 of A:", A_jan.get_row(0))
print("Col 0 of A^T:", AT.get_col(0))
print("They match:", A_jan.get_row(0) == AT.get_col(0))

### 3.8 Properties of the Transpose

The transpose satisfies four important algebraic properties:

1. $(A^T)^T = A$
2. $(A + B)^T = A^T + B^T$
3. $(cA)^T = cA^T$
4. $(AB)^T = B^T A^T$ &emsp; (note the **reversal of order**)

Let us verify each one computationally.

In [ ]:
X = Matrix.from_lists([[1, 2, 3], [4, 5, 6]])
Y = Matrix.from_lists([[7, 8, 9], [10, 11, 12]])
c = 3.0

print("Property 1: (A^T)^T = A")
prop1 = transpose(transpose(X))
print(f"  (X^T)^T == X? {prop1 == X}")
print()

print("Property 2: (A + B)^T = A^T + B^T")
lhs2 = transpose(add(X, Y))
rhs2 = add(transpose(X), transpose(Y))
print(f"  (X + Y)^T == X^T + Y^T? {lhs2 == rhs2}")
print()

print("Property 3: (cA)^T = cA^T")
lhs3 = transpose(scalar_mul(X, c))
rhs3 = scalar_mul(transpose(X), c)
print(f"  (cX)^T == cX^T? {lhs3 == rhs3}")
print()

print("Property 4: (AB)^T = B^T A^T")
Z = Matrix.from_lists([[1, 4], [2, 5], [3, 6]])
lhs4 = transpose(multiply(X, Z))
rhs4 = multiply(transpose(Z), transpose(X))
print(f"  (XZ)^T == Z^T X^T? {lhs4 == rhs4}")
print()
print("  (XZ)^T =")
print(lhs4)
print("  Z^T X^T =")
print(rhs4)

---

## 4. Verify --- Cross-Check with NumPy

Our from-scratch implementations should match NumPy exactly.
We verify each operation by converting to NumPy arrays and comparing results.

In [ ]:
def to_np(mat):
    """Convert our Matrix to a NumPy array."""
    return np.array(mat.to_lists())

def check_match(ours, theirs, label):
    """Compare our Matrix result against a NumPy array."""
    ours_np = to_np(ours)
    match = np.allclose(ours_np, theirs, atol=1e-9)
    status = "PASS" if match else "FAIL"
    print(f"[{status}] {label}")
    if not match:
        print(f"  Ours:   {ours_np}")
        print(f"  NumPy:  {theirs}")
    return match

In [ ]:
A_np = to_np(A_jan)
B_np = to_np(B_feb)
cost_np = to_np(cost)

check_match(
    add(A_jan, B_feb),
    A_np + B_np,
    "add(A, B) matches NumPy element-wise addition"
)

check_match(
    multiply(A_jan, cost),
    np.matmul(A_np, cost_np),
    "multiply(A, c) matches np.matmul"
)

prop_double_T = transpose(transpose(A_jan))
check_match(
    prop_double_T,
    A_np,
    "transpose(transpose(A)) == A"
)

AB = multiply(A_jan, cost)
check_match(
    transpose(AB),
    np.matmul(cost_np.T, A_np.T),
    "transpose(A*c) == c^T * A^T  [(AB)^T = B^T A^T]"
)

In [ ]:
P_np = to_np(P)
Q_np = to_np(Q)

PQ_np = np.matmul(P_np, Q_np)
QP_np = np.matmul(Q_np, P_np)

check_match(PQ, PQ_np, "multiply(P, Q) matches np.matmul(P, Q)")
check_match(QP, QP_np, "multiply(Q, P) matches np.matmul(Q, P)")

non_commutative = not np.allclose(PQ_np, QP_np)
print(f"\n[{'PASS' if non_commutative else 'FAIL'}] Non-commutativity: PQ != QP")

---

## 5. Visualize --- 2D Matrix Transformations

A $2 \times 2$ matrix $M$ defines a **linear transformation** on $\mathbb{R}^2$.
Applying $M$ to each vertex of the unit square shows how the transformation
distorts space.

We will visualize three fundamental transformations:

1. **Rotation** by $\theta = \pi/4$ (45 degrees)
2. **Scaling** --- stretch horizontally, compress vertically
3. **Shearing** --- slant along the x-axis

In [ ]:
unit_square = np.array([
    [0, 0],
    [1, 0],
    [1, 1],
    [0, 1],
    [0, 0],
])

theta = math.pi / 4
rotation = np.array([
    [math.cos(theta), -math.sin(theta)],
    [math.sin(theta),  math.cos(theta)],
])

scaling = np.array([
    [2.0, 0.0],
    [0.0, 0.5],
])

shearing = np.array([
    [1, 1],
    [0, 1],
])

transforms = [
    ("Rotation (45\u00b0)", rotation),
    ("Scaling (2x, 0.5x)", scaling),
    ("Shearing", shearing),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, (title, M) in zip(axes, transforms):
    transformed = unit_square @ M.T

    ax.fill(unit_square[:, 0], unit_square[:, 1],
            alpha=0.2, color='steelblue', label='Original')
    ax.plot(unit_square[:, 0], unit_square[:, 1],
            'o-', color='steelblue', markersize=5)

    ax.fill(transformed[:, 0], transformed[:, 1],
            alpha=0.2, color='tomato', label='Transformed')
    ax.plot(transformed[:, 0], transformed[:, 1],
            's-', color='tomato', markersize=5)

    ax.set_xlim(-1.5, 3.0)
    ax.set_ylim(-1.0, 2.0)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_title(title)
    ax.legend(fontsize=8)

    matrix_str = f"[[{M[0,0]:.2f}, {M[0,1]:.2f}]\n [{M[1,0]:.2f}, {M[1,1]:.2f}]]"
    ax.text(0.02, 0.98, f"M = {matrix_str}",
            transform=ax.transAxes, fontsize=7,
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

fig.suptitle('2D Linear Transformations Applied to the Unit Square',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 6. Exercises

Test your understanding with the following problems. Write your solutions
in the code cells provided.

### Exercise 1 (Easy)

Let

$$A = \begin{bmatrix} 1 & 2 \\ 3 & 4 \end{bmatrix}, \quad B = \begin{bmatrix} 5 & 6 \\ 7 & 8 \end{bmatrix}$$

Compute $AB$ and $BA$ using `multiply`. Verify that $AB \neq BA$.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium)

Implement a function `power(A, n)` that computes $A^n$ using repeated
multiplication ($A^0 = I$, $A^n = A \cdot A^{n-1}$).

Test it with the **rotation matrix**

$$R = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}, \quad \theta = \frac{\pi}{4}$$

Since $R$ rotates by 45 degrees, $R^8$ should rotate by $8 \times 45 = 360$ degrees,
which means $R^8 = I_2$. Verify this.

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge)

There are two natural loop orderings for computing the $(i, j)$ entry of the
product $C = AB$ of two $n \times n$ matrices:

- **i-j-k order:** For each row $i$ and column $j$, accumulate
  $c_{ij} = \sum_k a_{ik} b_{kj}$.
- **i-k-j order:** For each row $i$ and index $k$, scatter
  $c_{ij} \mathrel{+}= a_{ik} b_{kj}$ for all $j$.

Implement both orderings and time them on $200 \times 200$ random matrices.
Which is faster, and **why**? (Hint: think about memory access patterns.)

In [ ]:
# YOUR CODE HERE


---

## Summary

| Concept | Key Takeaway |
|---|---|
| **Matrix addition** | Entry-by-entry; requires same dimensions |
| **Scalar multiplication** | Multiply every entry by the scalar |
| **Matrix multiplication** | $(AB)_{ij} = $ dot product of row $i$ of $A$ with column $j$ of $B$; requires inner dimensions to match |
| **Non-commutativity** | In general $AB \neq BA$ |
| **Identity matrix** | $AI = IA = A$ for all conformable $A$ |
| **Transpose** | $(A^T)_{ij} = a_{ji}$; row $i$ becomes column $i$ |
| **Transpose of a product** | $(AB)^T = B^T A^T$ --- order reverses |